# La Liga season finish probabilities (simple model)

This notebook builds a simple Poisson-based model from finished La Liga matches,
simulates the rest of the season, and renders a position probability table.

Requirements:
- Set `FOOTBALL_DATA_API_KEY` in your environment (from https://www.football-data.org/).


In [5]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from dotenv import load_dotenv

from standing_prediction.data import FootballDataClient, fetch_standings, fetch_matches
from standing_prediction.model import fit_strengths, predict_fixtures
from standing_prediction.simulate import simulate_season


In [6]:
season = 2025  # season start year
load_dotenv(".env")
client = FootballDataClient()

standings = fetch_standings(client, competition="PD", season=season)
matches_finished = fetch_matches(client, competition="PD", season=season, status="FINISHED")
matches_scheduled = fetch_matches(client, competition="PD", season=season, status="SCHEDULED,TIMED")

standings.head()


,position,team,played,points,goals_for,goals_against,goal_diff
0,1,FC Barcelona,25,61,67,25,42
1,2,Real Madrid CF,25,60,54,21,33
2,3,Villarreal CF,25,51,47,27,20
3,4,Club Atlético de Madrid,25,48,42,23,19
4,5,Real Betis Balompié,25,42,40,30,10


In [7]:
model = fit_strengths(matches_finished)
fixture_probs = predict_fixtures(matches_scheduled, model)

fixture_probs.head()


,utcDate,homeTeam,awayTeam,p_home_win,p_draw,p_away_win
0,2026-02-27 20:00:00+00:00,Levante UD,Deportivo Alavés,0.386249,0.292002,0.321749
1,2026-02-28 13:00:00+00:00,Rayo Vallecano de Madrid,Athletic Club,0.391387,0.301359,0.307254
2,2026-02-28 15:15:00+00:00,FC Barcelona,Villarreal CF,0.655939,0.178476,0.165584
3,2026-02-28 17:30:00+00:00,RCD Mallorca,Real Sociedad de Fútbol,0.377722,0.240333,0.381944
4,2026-02-28 20:00:00+00:00,Real Oviedo,Club Atlético de Madrid,0.108785,0.224634,0.666581


In [8]:
counts = simulate_season(fixture_probs, standings, n_sim=5000, seed=42)
position_pct = counts.div(counts.sum(axis=1), axis=0) * 100

teams_ordered = standings.sort_values("position")["team"]
position_pct = position_pct.loc[teams_ordered]

position_pct.head()


,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
FC Barcelona,64.32,34.78,0.90,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Real Madrid CF,35.54,62.74,1.68,0.04,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Villarreal CF,0.14,2.14,75.80,20.28,1.62,0.02,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Club Atlético de Madrid,0.00,0.34,20.96,69.68,8.58,0.38,0.06,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Real Betis Balompié,0.00,0.00,0.66,9.56,76.02,10.56,2.28,0.74,0.16,0.02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np

greens = plt.cm.Greens
green_cmap = LinearSegmentedColormap.from_list(
    "Greens_soft",
    greens(np.linspace(0.05, 0.72, 256))  # trims the darkest tail
)

vmax = 20


def zero_style(val):
    if val < 0.005:
        return (
            "background-color: white !important;"
        )
    return ""


position_pct.style\
    .background_gradient(
        cmap=green_cmap,
        vmin=0,
        vmax=vmax
    )\
    .map(zero_style)\
    .format("{:.2f}%")\
    .set_table_styles([
        {"selector": "th", "props": [
            ("background-color", "#e6edf4"),
            ("color", "#333"),
            ("text-align", "center"),
            ("font-family", "Inter, Roboto, Arial, sans-serif"),
            ("font-size", "13px"),
            ("font-weight", "600")
        ]},

        # <-- LEFT align only the "position" column header
        {"selector": "th.col_heading", "props": [
            ("text-align", "centre")
        ]},


        {"selector": "th.row_heading", "props": [
            ("text-align", "left"),
            ("font-size", "13px"),
            ("font-weight", "600"),
            ("white-space", "nowrap"),
            ("max-width", "250px"),
            ("overflow", "hidden"),
            ("text-overflow", "ellipsis")
        ]},

        {"selector": "tr:nth-child(odd) th.row_heading", "props": [
            ("background-color", "#fbfcfe")
        ]},
        {"selector": "tr:nth-child(even) th.row_heading", "props": [
            ("background-color", "#e6edf4")
        ]},

        {"selector": "td", "props": [
            ("text-align", "center"),
            ("font-family", "Inter, Roboto, Arial, sans-serif"),
            ("font-size", "12px"),
            ("font-weight", "500"),
            ("color", "#000")
        ]}
    ])

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
FC Barcelona,64.32%,34.78%,0.90%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Real Madrid CF,35.54%,62.74%,1.68%,0.04%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Villarreal CF,0.14%,2.14%,75.80%,20.28%,1.62%,0.02%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Club Atlético de Madrid,0.00%,0.34%,20.96%,69.68%,8.58%,0.38%,0.06%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Real Betis Balompié,0.00%,0.00%,0.66%,9.56%,76.02%,10.56%,2.28%,0.74%,0.16%,0.02%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
RC Celta de Vigo,0.00%,0.00%,0.00%,0.32%,9.74%,50.98%,21.64%,9.82%,4.60%,1.66%,0.84%,0.22%,0.12%,0.06%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
RCD Espanyol de Barcelona,0.00%,0.00%,0.00%,0.02%,1.02%,9.54%,18.90%,21.54%,18.36%,13.48%,7.98%,4.18%,2.36%,1.48%,0.66%,0.30%,0.16%,0.02%,0.00%,0.00%
Athletic Club,0.00%,0.00%,0.00%,0.02%,0.32%,3.54%,7.10%,12.30%,17.18%,18.34%,14.36%,9.16%,7.22%,4.76%,3.02%,1.64%,0.76%,0.22%,0.06%,0.00%
CA Osasuna,0.00%,0.00%,0.00%,0.06%,1.74%,14.18%,26.60%,21.14%,14.94%,9.28%,5.36%,2.98%,1.50%,1.22%,0.54%,0.32%,0.08%,0.06%,0.00%,0.00%
Real Sociedad de Fútbol,0.00%,0.00%,0.00%,0.02%,0.94%,9.66%,18.34%,20.82%,18.80%,12.48%,6.82%,5.46%,2.76%,2.02%,1.04%,0.54%,0.16%,0.12%,0.02%,0.00%
